# M2.2: Price formation — merit-order effect, cannibalisation, negative prices, carbon pricing

## Objectives

At the end of this notebook you should be able to:

- clear a market for a **whole year** hour by hour, and check it against PyPSA
- explain the **merit-order effect**: why adding wind and solar lowers prices
- compute **capture prices** and **capture rates**, and explain **cannibalisation** — why solar loses value faster than wind
- explain where **negative prices** come from, and how a support rule changes them
- derive the **fuel-switching CO₂ price** between coal and gas, and confirm it by simulation

M2.1 cleared one hour. Prices that matter for investment are yearly patterns: when are they high or low, and which technologies produce in which hours? The system here is **synthetic** but Croatia-sized (18.4 TWh/year demand); real Croatian prices follow in M2.3. All fuel and technology numbers are **illustrative**.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### A synthetic year

Hourly demand with seasonal, daily and weekly patterns; wind with multi-day weather spells (an autoregressive process, as in M1.5); solar from the sun's daily arc times a random daily cloudiness; run-of-river with a spring maximum from snowmelt:

In [2]:
rng = np.random.default_rng(7)
idx = pd.date_range("2025-01-01", periods=8760, freq="h")
hour, doy = idx.hour.to_numpy(), idx.dayofyear.to_numpy()

load = (2050 + 350 * np.cos(2 * np.pi * (doy - 15) / 365) + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0)
        - 150 * (idx.dayofweek.to_numpy() >= 5) + rng.normal(0, 60, 8760))           # MW
w = np.zeros(8760)
shocks = rng.normal(0, 1, 8760)
for t in range(1, 8760):
    w[t] = 0.97 * w[t - 1] + 0.24 * shocks[t]
wind_cf = np.clip(0.27 + 0.18 * w + 0.06 * np.cos(2 * np.pi * (doy - 15) / 365), 0, 0.95)
sun = np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * (0.65 + 0.35 * np.sin(2 * np.pi * (doy - 80) / 365))
cloud = np.clip(1 - 0.5 * rng.beta(2, 3, 365).repeat(24), 0.2, 1)
solar_cf = np.clip(sun * cloud * 0.95, 0, 1)
ror_cf = np.clip(0.55 + 0.25 * np.sin(2 * np.pi * (doy - 60) / 365), 0.2, 0.9)

print(f"demand {load.sum() / 1e6:.1f} TWh/year, {load.min():,.0f}-{load.max():,.0f} MW | "
      f"capacity factors: wind {wind_cf.mean():.3f}, solar {solar_cf.mean():.3f}, run-of-river {ror_cf.mean():.3f}")

demand 18.4 TWh/year, 1,398-2,834 MW | capacity factors: wind 0.286, solar 0.169, run-of-river 0.550


### Who offers at what price

Each plant offers its available capacity at its marginal cost. For a thermal plant that is fuel cost plus CO₂ cost, both per MWh of *electricity*: **MC = (fuel price + CO₂ price × emission factor) / efficiency**. Two technologies offer **below zero**:

- **Wind and solar with a feed-in premium** of 20 €/MWh receive the premium only when they produce, so producing at −20 €/MWh still breaks even: they offer at −20.
- **Combined heat and power (CHP)** plants that must run to supply heat or industrial steam; here 150 MW must-run all year, offered at −30 €/MWh.

In [3]:
COAL = {"fuel": 12.0, "eff": 0.38, "ef": 0.34}     # EUR/MWh_th, -, tCO2/MWh_th (illustrative)
GAS = {"fuel": 35.0, "eff": 0.55, "ef": 0.20}
OCGT = {"fuel": 35.0, "eff": 0.38, "ef": 0.20}


def thermal_mc(plant, co2):
    """Marginal cost (EUR/MWh_el) of a thermal plant at CO2 price co2 (EUR/t)."""
    return (plant["fuel"] + co2 * plant["ef"]) / plant["eff"]


def fleet(wind_MW=1000, solar_MW=1000, co2=70.0, res_bid=-20.0):
    """Offers as (name, available MW per hour, price EUR/MWh)."""
    return [("solar", solar_MW * solar_cf, res_bid), ("wind", wind_MW * wind_cf, res_bid),
            ("chp_must_run", np.full(8760, 150.0), -30.0), ("run_of_river", 500 * ror_cf, 0.0),
            ("nuclear_share", np.full(8760, 350.0), 12.0), ("coal", np.full(8760, 600.0), thermal_mc(COAL, co2)),
            ("ccgt", np.full(8760, 900.0), thermal_mc(GAS, co2)), ("imports", np.full(8760, 800.0), thermal_mc(GAS, co2) + 15),
            ("ocgt", np.full(8760, 400.0), thermal_mc(OCGT, co2))]


pd.DataFrame([(n, c.mean(), p) for n, c, p in fleet()], columns=["offer", "mean_available_MW", "price_EUR_MWh"]).round(1)

           offer  mean_available_MW  price_EUR_MWh
0          solar              169.3          -20.0
1           wind              285.9          -20.0
2   chp_must_run              150.0          -30.0
3   run_of_river              275.0            0.0
4  nuclear_share              350.0           12.0
5           coal              600.0           94.2
6           ccgt              900.0           89.1
7        imports              800.0          104.1
8           ocgt              400.0          128.9

### Clearing 8,760 hours at once

Demand here is inelastic, so each hour's price is the offer price of the unit that covers the last MW (M1.2's merit order). Sorting the offers once and taking a cumulative sum per hour clears the whole year in one vectorised step:

In [4]:
def clear_year(offers, demand):
    """Hourly price (EUR/MWh) and dispatch (MW) for inelastic demand."""
    names = [o[0] for o in offers]
    cap = np.vstack([o[1] for o in offers])
    price = np.array([o[2] for o in offers])
    order = np.argsort(price, kind="stable")
    cap, price = cap[order], price[order]
    cum = np.cumsum(cap, axis=0)
    marginal = (cum < demand).sum(axis=0)                     # first unit whose cumulative capacity covers demand
    dispatch = np.clip(np.minimum(cap, demand - (cum - cap)), 0, None)
    return price[marginal], pd.DataFrame(dispatch.T, index=idx, columns=[names[i] for i in order])


base = fleet()
price, dispatch = clear_year(base, load)
assert np.allclose(dispatch.sum(axis=1), load), "demand met in every hour"

n = pypsa.Network()
n.set_snapshots(idx)
n.add("Bus", "HR")
for name, cap, p in base:
    n.add("Generator", name, bus="HR", p_nom=cap.max(), p_max_pu=cap / cap.max(), marginal_cost=p)
n.add("Load", "load", bus="HR", p_set=load)
n.optimize(solver_name="highs", log_to_console=False, progress=False)
pypsa_price = n.buses_t.marginal_price["HR"].to_numpy()
assert np.allclose(pypsa_price, price), "identical in every hour"
print(f"own clearing and PyPSA agree in all {len(idx):,} hours (largest difference {np.abs(pypsa_price - price).max():.1e} EUR/MWh)")
print(f"average price {price.mean():.2f} EUR/MWh, range {price.min():.1f} to {price.max():.1f}, negative hours: {(price < 0).sum()}")

own clearing and PyPSA agree in all 8,760 hours (largest difference 0.0e+00 EUR/MWh)
average price 90.96 EUR/MWh, range 0.0 to 104.1, negative hours: 0


Your own vectorised clearing and PyPSA's full-year optimisation agree in **every one of the 8,760 hours**. The vectorised version needs no solver at all — useful for quick scenario sweeps — while PyPSA takes over as soon as storage, networks or investment couple the hours (Module 03).

### The merit-order effect and cannibalisation

Zero-cost renewables push the supply curve to the right (M2.1's challenge), so the price falls. But *when* does it fall? In exactly the hours when the renewables produce. The **capture price** — the average price a technology actually earns, weighted by its output — tells you:

**capture price = Σ_t price_t · generation_t / Σ_t generation_t,  capture rate = capture price / average price**

In [5]:
def capture_price(price, generation):
    return float((price * generation).sum() / generation.sum())


rows = []
for tech in ["wind", "solar"]:
    for mw in [500, 1000, 2000, 3000]:
        kwargs = {"wind_MW": 1000, "solar_MW": 1000, f"{tech}_MW": mw}
        p, d = clear_year(fleet(**kwargs), load)
        cp = capture_price(p, d[tech])
        rows.append({"technology": tech, "capacity_MW": mw, "share_of_demand": d[tech].sum() / load.sum(),
                     "average_price": p.mean(), "capture_price": cp, "capture_rate": cp / p.mean(),
                     "negative_hours": int((p < 0).sum())})
cannibal = pd.DataFrame(rows)
cannibal.round(3)

  technology  capacity_MW  share_of_demand  average_price  capture_price  \
0       wind          500            0.068         93.103         93.041   
1       wind         1000            0.136         90.956         90.118   
2       wind         2000            0.272         81.683         76.436   
3       wind         3000            0.403         64.090         49.822   
4      solar          500            0.040         92.564         91.713   
5      solar         1000            0.081         90.956         86.261   
6      solar         2000            0.161         81.151         53.336   
7      solar         3000            0.234         71.806         29.376   

   capture_rate  negative_hours  
0         0.999               0  
1         0.991               0  
2         0.936              18  
3         0.777             436  
4         0.991               0  
5         0.948               0  
6         0.657             102  
7         0.409             758  

In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
for tech, g in cannibal.groupby("technology"):
    ax.plot(100 * g.share_of_demand, g.capture_rate, "o-", label=tech)
ax.axhline(1, color="grey", lw=0.8, ls=":")
ax.set(xlabel="share of annual demand (%)", ylabel="capture rate (capture / average price)",
       title="Cannibalisation: the more you build, the less each MWh earns")
ax.legend()
plt.show()

Both curves fall, and solar falls far faster. With 1,000 MW each, solar already earns only 95 % of the average price; at 3,000 MW of solar (about 24 % of demand) its capture rate drops to **0.41**, while wind at a larger share keeps 0.78. Every solar panel produces at the same moments — sunny middays — so its output pushes down exactly the prices it is paid. Wind blows at different times across the day and with weather systems, and spreads its output over more hours.

This is **cannibalisation**, and it is the central economic problem of a high-renewable system: the cheaper energy gets, the harder it is to finance more of it from market prices alone. Its remedies are flexibility — storage, flexible demand, interconnection — that moves demand into the cheap hours (the exercise and Module 03).

### Negative prices

The base system has no negative prices. With 2,500 MW of solar, sunny low-demand hours bring subsidised solar and wind plus must-run CHP above demand — and the price falls to their offer price:

In [7]:
p_neg, d_neg = clear_year(fleet(solar_MW=2500), load)
neg = pd.Series(p_neg < 0, index=idx)
print(f"2,500 MW solar: {neg.sum()} negative hours, lowest price {p_neg.min():.0f} EUR/MWh")
share_midday_spring = neg[(idx.hour >= 10) & (idx.hour <= 15) & (idx.month.isin([4, 5, 6]))].sum() / neg.sum()
print(f"of these, {share_midday_spring:.0%} fall between 10:00 and 15:59 in April-June")

p_rule, _ = clear_year(fleet(solar_MW=2500, res_bid=0.0), load)
print(f"if support is suspended during negative prices (renewables bid 0): {(p_rule < 0).sum()} negative hours, "
      f"lowest price {p_rule.min():.0f} EUR/MWh")

2,500 MW solar: 403 negative hours, lowest price -20 EUR/MWh
of these, 53% fall between 10:00 and 15:59 in April-June
if support is suspended during negative prices (renewables bid 0): 0 negative hours, lowest price 0 EUR/MWh


More than half of the negative hours fall in spring middays (10:00–15:59, April–June), a window of only about 6 % of the year — strong sun, snowmelt-driven run-of-river, mild temperatures and lower demand. Their depth is set by the support scheme: renewables paid per MWh are willing to *pay* up to their premium to keep producing. When support is suspended during negative-price hours — a rule European support schemes increasingly include — renewables offer at 0 €/MWh, curtail themselves instead, and negative prices disappear in this system: the must-run CHP alone never exceeds demand.

Negative prices are a **signal**, not a malfunction: they pay flexible consumers, storage and exporters to absorb surplus, and tell inflexible producers to become flexible.

### Carbon pricing and fuel switching

With a low CO₂ price coal (cheap fuel, high emissions) is cheaper than gas; with a high one, gas is. The **switching price** is where their marginal costs are equal:

(12 + 0.34·c) / 0.38 = (35 + 0.20·c) / 0.55  →  c = (35/0.55 − 12/0.38) / (0.34/0.38 − 0.20/0.55)

In [8]:
switch = (GAS["fuel"] / GAS["eff"] - COAL["fuel"] / COAL["eff"]) / (COAL["ef"] / COAL["eff"] - GAS["ef"] / GAS["eff"])
print(f"predicted switching price: {switch:.2f} EUR/t")

rows = []
for co2 in [40, 50, 59, 60, 61, 62, 70, 80]:
    p, d = clear_year(fleet(co2=co2), load)
    emissions = (d.coal * COAL["ef"] / COAL["eff"] + (d.ccgt + d.ocgt) * GAS["ef"] / GAS["eff"]).sum() / 1e6
    rows.append({"co2_EUR_t": co2, "coal_TWh": d.coal.sum() / 1e6, "ccgt_TWh": d.ccgt.sum() / 1e6,
                 "domestic_emissions_Mt": emissions, "average_price": p.mean()})
switching = pd.DataFrame(rows).set_index("co2_EUR_t")
assert switching.loc[60, "coal_TWh"] > 2 * switching.loc[61, "coal_TWh"], "the switch happens between 60 and 61 EUR/t"
switching.round(2)

predicted switching price: 60.36 EUR/t


           coal_TWh  ccgt_TWh  domestic_emissions_Mt  average_price
co2_EUR_t                                                          
40             4.71      2.89                   5.27          75.31
50             4.71      2.89                   5.27          80.19
59             4.71      2.89                   5.27          84.57
60             4.71      2.89                   5.27          85.06
61             1.26      6.34                   3.44          85.61
62             1.26      6.34                   3.44          86.21
70             1.26      6.34                   3.44          90.96
80             1.26      6.34                   3.44          96.89

The simulation confirms the hand calculation: coal generation drops sharply between 60 and 61 €/t, exactly where the prediction (60.36 €/t) says. Below it, raising the CO₂ price only raises electricity prices — nobody changes behaviour; above it, gas displaces coal and domestic emissions fall by about a third (5.3 to 3.4 Mt) in one step. With many plants of different efficiencies, real systems switch gradually over a band of CO₂ prices rather than at one point (Maths-2's tax-vs-cap challenge).

Note what a CO₂ price does to the *electricity* price: the average rises steadily with it, because the marginal plant passes its carbon cost through. Who pays for carbon is ultimately the consumer — and who gains are the low-carbon plants, whose inframarginal rent grows.

## Check your understanding

**Renewables**

1. Why does wind keep a higher capture rate than solar at the same share of demand? Name a system change that would help solar's capture rate most.
2. A wind farm developer quotes a levelised cost of 60 €/MWh. With 3,000 MW of wind already built, is a new wind farm profitable at market prices in this system? What extra information would you want?

**Prices and carbon**

1. In which months would you expect the most negative-price hours in Croatia, and why does run-of-river hydro matter?
2. Between 40 and 60 €/t the CO₂ price changes nothing in the dispatch. What does it change, and who benefits?